# Day 1.4 — Tool Calling

A model can generate text, but it cannot directly execute your Python functions. We will let it **request** one safe calculator tool.

```text
User → model requests tool → Python validates and executes → result returns to model
```

By the end, you can define a tool schema, inspect a model request, execute it in Python, and return the observation.

## Before you begin

### Learning outcomes

Distinguish a model tool request from host validation and Python execution.

Architecture reference: [D03](../../diagrams/source/day_01.md).

### Expected observation

The model returns a name and arguments; the calculator runs only after validation in application code.


In [ ]:
import ast,json,operator,os
from types import SimpleNamespace
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel,Field,ValidationError
load_dotenv(); api_key=os.getenv("OPENROUTER_API_KEY")
client=OpenAI(base_url="https://openrouter.ai/api/v1",api_key=api_key) if api_key else None
COURSE_MODEL=os.getenv("OPENROUTER_MODEL","openai/gpt-oss-120b")
print("Route:","OpenRouter" if client else "mock fallback")


## Build a safe calculator

Never use unrestricted `eval()` on model-generated input. This deliberately small evaluator accepts numbers and basic arithmetic operators only.

In [ ]:
BINARY = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv}
UNARY = {ast.UAdd: operator.pos, ast.USub: operator.neg}

def evaluate_node(node):
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return float(node.value)
    if isinstance(node, ast.BinOp) and type(node.op) in BINARY:
        return BINARY[type(node.op)](evaluate_node(node.left), evaluate_node(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in UNARY:
        return UNARY[type(node.op)](evaluate_node(node.operand))
    raise ValueError("Only basic arithmetic is allowed")

def calculator(expression: str) -> str:
    value = evaluate_node(ast.parse(expression, mode="eval").body)
    return str(int(value)) if value.is_integer() else str(value)

calculator("12 * 7")

## Describe the tool to the model with a schema

In [ ]:
class CalculatorArguments(BaseModel):
    expression: str = Field(min_length=1, max_length=100)

calculator_tool = {
    "type": "function",
    "function": {
        "name": "calculator",
        "description": "Evaluate basic arithmetic instead of calculating mentally.",
        "parameters": {
            "type": "object",
            "properties": {"expression": {"type": "string"}},
            "required": ["expression"],
            "additionalProperties": False,
        },
    },
}

## Ask the model

The returned `tool_calls` value is a request—not evidence that anything has executed.

In [ ]:
messages=[{"role":"user","content":"What is 12 * 7? Use the calculator."}]
if client:
    first=client.chat.completions.create(model=COURSE_MODEL,messages=messages,tools=[calculator_tool],max_tokens=400,
        extra_body={"reasoning":{"effort":"low","exclude":False},"provider":{"require_parameters":True}})
    assistant_message=first.choices[0].message
else:
    call=SimpleNamespace(id="mock-calculator",function=SimpleNamespace(name="calculator",arguments='{"expression":"12 * 7"}'))
    assistant_message=SimpleNamespace(tool_calls=[call],model_dump=lambda **kwargs:{"role":"assistant","content":"","tool_calls":[{"id":call.id,"type":"function","function":{"name":"calculator","arguments":call.function.arguments}}]})
assistant_message.tool_calls


## Validate and execute in Python

In [ ]:
call = assistant_message.tool_calls[0]
arguments = CalculatorArguments.model_validate_json(call.function.arguments)
tool_output = calculator(arguments.expression)
print(call.function.name, arguments.expression, tool_output)

## Return the observation to the model

Append the assistant's original tool request and a tool-role result so the model receives the complete sequence.

In [ ]:
messages.append(assistant_message.model_dump(exclude_none=True))
messages.append({"role":"tool","tool_call_id":call.id,"content":tool_output})
if client:
    final=client.chat.completions.create(model=COURSE_MODEL,messages=messages,tools=[calculator_tool],max_tokens=300,
        extra_body={"reasoning":{"effort":"low","exclude":True}})
    final_text=final.choices[0].message.content
else:
    final_text=f"The calculator result is {tool_output}."
print(final_text)


## Break and inspect

Try malformed arguments and `__import__('os').getcwd()`. The schema checks shape; the tool implementation enforces what operations are permitted. Both layers matter.

## Exercise and checkpoint

Add a `convert_celsius_to_fahrenheit` tool with one numeric argument. Inspect the request before executing it.

We now have one complete tool interaction. The limitation is that the code assumes exactly one request and one tool call. A manual agent loop generalizes it next.

## Your turn

Send an unsupported argument and prove the function is not executed.

## Recap

The model requests; the host validates, authorizes, and executes.
